# Übung 5: Wie verlässlich ist das Ergebnis? Validierung über die Zeit
**Internet of Production – KI, Woche 2** · ca. 35 min · Arbeit in Zweiergruppen

**Roter Faden:** Übung 4 lieferte eine Zahl je Modell. Die Daten sind aber eine **Zeitreihe mit nur 34 Blockaden**. Kann man einer einzigen Aufteilung trauen? In dieser Übung prüfen Sie, wie stark das Ergebnis von der Validierung abhängt.

**Fragen**
1. Warum haben wir nach Zeit und nicht zufällig aufgeteilt? Was ändert sich bei zufälliger Aufteilung?
2. Wie stark schwankt das Ergebnis, wenn die Aufteilung durch die Zeit wandert?
3. Helfen zusätzliche Merkmale (Temperaturen), oder passen sie nur Rauschen an?

Jedes `___` ist eine Lücke.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes: warn 6 hours ahead
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 of week 1 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values)
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages, idx)
    mtb = np.where(nxt < len(blockages), (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    p['Temp_Extruder_m'] = p['Temp_Extruder'].rolling(120).mean()
    p['Temp_Bed_m'] = p['Temp_Bed'].rolling(120).mean()
    return p.dropna(subset=FEATURES + ['Temp_Extruder_m']).iloc[::10], pd.to_datetime(blockages)

def report(name, y_true, y_pred, score=None):
    auc = f'   AUC {roc_auc_score(y_true, score):5.3f}' if score is not None else ''
    print(f'{name:36s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}{auc}')

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)                      # split by time: first 70 % train, last 30 % test
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[FEATURES], train['y'], test[FEATURES], test['y']
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1; train {len(train)}, test {len(test)}")

from sklearn.model_selection import train_test_split, TimeSeriesSplit

## Schritt 1: Zufällige Aufteilung gegen Aufteilung nach Zeit
Benachbarte Stichproben liegen nur 10 Minuten auseinander, sind fast identisch und haben meist dasselbe Label. Bei **zufälliger** Aufteilung hat eine Teststichprobe daher eine fast gleiche Kopie im Training. Vergleichen Sie drei Modelle mit beiden Aufteilungen (Recall auf den Testdaten).

In [ ]:
models = {
    '1-nearest neighbour': lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
    'tree, no limit': lambda: DecisionTreeClassifier(random_state=0),
    'tree, depth 3, balanced': lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, class_weight='balanced', random_state=0),
    'KNN, k = 25': lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
}
Xr_train, Xr_test, yr_train, yr_test = train_test_split(data[FEATURES], data['y'], test_size=___, random_state=0, stratify=data['y'])

rows = []
for name, make in models.items():
    m_time = make().fit(X_train, y_train)
    m_rand = make().fit(Xr_train, yr_train)
    rows.append({'model': name,
                 'recall, time split': recall_score(y_test, m_time.predict(X_test)),
                 'recall, random split': recall_score(yr_test, m_rand.predict(Xr_test)),
                 'precision, time split': precision_score(y_test, m_time.predict(X_test), zero_division=0),
                 'precision, random split': precision_score(yr_test, m_rand.predict(Xr_test), zero_division=0)})
display(pd.DataFrame(rows).set_index('model').round(3))

**Bei welchen Modellen sieht die zufällige Aufteilung besser aus als die Aufteilung nach Zeit? Warum profitieren flexible Modelle am meisten? Was würde ein Kunde vom Ergebnis erwarten?**

*Ihre Antwort:*

## Schritt 2: Die Aufteilung durch die Zeit schieben
`TimeSeriesSplit` trainiert mit der Vergangenheit und testet auf dem nächsten Block, fünfmal mit wachsendem Trainingsdatensatz. Zählen Sie, wie viele **Blockaden** jeder Testblock enthält.

In [ ]:
def evaluate_over_time(make, features, n_splits=___):
    rows = []
    for i, (a, bb) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(data), 1):
        tr, te = data.iloc[a], data.iloc[bb]
        m = make().fit(tr[features], tr['y'])
        n_blk = int(((blockages >= te.index[0]) & (blockages <= te.index[-1])).sum())
        rows.append({'fold': i, 'test from': te.index[0].date(), 'blockages in test': n_blk,
                     'recall': recall_score(te['y'], m.predict(te[features])),
                     'precision': precision_score(te['y'], m.predict(te[features]), zero_division=0),
                     'AUC': roc_auc_score(te['y'], m.predict_proba(te[features])[:, 1])})
    return pd.DataFrame(rows).set_index('fold')

folds = evaluate_over_time(models['tree, depth 3, balanced'], FEATURES)
display(folds.round(3))
print('mean recall', folds['recall'].mean().round(3), ' min', folds['recall'].min().round(3), ' max', folds['recall'].max().round(3))

**Wie stark schwankt der Recall zwischen den Blöcken, und wie viele Blockaden stehen hinter jedem Block? Was bedeutet das für den Satz „das Modell findet 80 % der Blockaden“?**

*Ihre Antwort:*

## Schritt 3: Helfen Temperaturen?
Fügen Sie die mittlere Düsen- und Bett-Temperatur der letzten 2 Stunden als Merkmale hinzu und wiederholen Sie die Bewertung über die Zeit. Beurteilen Sie die Änderung vor dem Hintergrund der Schwankung, die Sie eben gesehen haben.

In [ ]:
features_t = FEATURES + [___, ___]
folds_t = evaluate_over_time(models['tree, depth 3, balanced'], features_t)
cmp = pd.DataFrame({'AUC energy only': folds['AUC'], 'AUC + temperatures': folds_t['AUC']})
cmp['difference'] = cmp['AUC + temperatures'] - cmp['AUC energy only']
display(cmp.round(3)); print(cmp.mean().round(3).to_dict())

# physical plausibility: mean temperatures in the hours before a blockage
dd = data.copy()
dd['group'] = np.where(dd['y'] == 1, 'within 6 h before blockage', 'other')
display(dd.groupby('group')[['E120', 'Temp_Extruder_m', 'Temp_Bed_m']].mean().round(3))

**Würden Sie die Temperaturen in das Modell aufnehmen? Berücksichtigen Sie die Größe des Gewinns, die Zahl der Ereignisse und ob es eine physikalische Erklärung gibt.**

*Ihre Antwort:*

## Schritt 4: Zusammenfassung

**In vier Sätzen: Was sagen Sie dem Vorfertigungsleiter zur Güte der Warnung, und was empfehlen Sie für die nächsten Schritte (Daten, Validierung, Modell)?**

*Ihre Antwort:*